# Energy Forecast — Prediction Performance (last 30 days)

Pulls live data from Home Assistant via SMB, fetches weather from Open-Meteo, and renders accuracy charts for the last 30 days.

**Pre-requisite:** `SMB_PASSWORD` environment variable must be set before starting the kernel.


In [ ]:
from __future__ import annotations

import io
import json
import os
from datetime import date, timedelta

import altair as alt
import pandas as pd
import requests
import yaml
from smb.SMBConnection import SMBConnection


In [ ]:
# ── Credentials ──────────────────────────────────────────────────────────────
SMB_USER = os.getenv("SMB_USER", "martin")
SMB_PASSWORD = os.getenv("SMB_PASSWORD")
if not SMB_PASSWORD:
    raise RuntimeError(
        "SMB_PASSWORD environment variable is not set. "
        "Set it before starting the kernel: export SMB_PASSWORD=<password>"
    )

# ── Constants ─────────────────────────────────────────────────────────────────
HA_HOST = "homeassistant"
SMB_SHARE = "addon_configs"
AD_BASE = "a0d7b954_appdaemon/apps"
FORECAST_REMOTE = f"{AD_BASE}/energy_forecast"

TZ = "Europe/Zurich"
CUTOFF_DAYS = 30
EV_THRESHOLD_KWH = 7.0  # matches EV_CHARGING_THRESHOLD_KWH in const.py

WEEKDAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

print("Config OK — SMB_PASSWORD is set")

## 1 · Fetch data from Home Assistant (SMB)

In [ ]:
def _smb_read(remote_path: str) -> bytes:
    conn = SMBConnection(SMB_USER, SMB_PASSWORD, "notebook", HA_HOST, use_ntlm_v2=True)
    if not conn.connect(HA_HOST, 445):
        raise ConnectionError(f"SMB connection to {HA_HOST}:445 failed")
    try:
        buf = io.BytesIO()
        conn.retrieveFile(SMB_SHARE, remote_path, buf)
        return buf.getvalue()
    finally:
        conn.close()


# Pull files
print("Fetching pred_history.json …")
_pred_raw = json.loads(_smb_read(f"{FORECAST_REMOTE}/pred_history.json"))
print(f"  pred entries : {len(_pred_raw.get('pred', {}))}")
print(f"  actual entries: {len(_pred_raw.get('actuals', {}))}")

print("Fetching apps.yaml …")
_apps_yaml = yaml.safe_load(_smb_read(f"{AD_BASE}/apps.yaml"))

if not _pred_raw.get("pred"):
    print("WARNING: pred_history.json has 0 prediction entries — charts will be empty.")
    print("The app needs at least one update cycle on the HA system to populate this file.")
